In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

from featuregraph.utils._arc_agi import get_training_pairs, state_training_cycle, test_cycle, derive_state_instruction_layout

In [2]:
training_directory = Path("../../ARC-AGI-2/data/training")

task_paths = sorted(training_directory.glob("*.json"))[:25]

challenges = {}

for task_path in task_paths:
    with task_path.open(encoding="utf-8") as file:
        challenges[task_path.stem] = json.load(file)

len(challenges)

representative_task_id = "007bbfb7"
representative_task = challenges[representative_task_id]

shape_rows = []

for pair_type in ("train", "test"):
    for pair_index, pair in enumerate(
        representative_task[pair_type]
    ):
        input_grid = np.asarray(pair["input"])

        output_grid = (
            np.asarray(pair["output"])
            if "output" in pair
            else None
        )

        shape_rows.append(
            {
                "pair_type": pair_type,
                "pair_index": pair_index,
                "input_shape": input_grid.shape,
                "output_shape": (
                    output_grid.shape
                    if output_grid is not None
                    else None
                ),
                "output_block_rows": (
                    output_grid.shape[0] // input_grid.shape[0]
                    if output_grid is not None
                    else None
                ),
                "output_block_columns": (
                    output_grid.shape[1] // input_grid.shape[1]
                    if output_grid is not None
                    else None
                ),
            }
        )

In [3]:
training_pairs = get_training_pairs(representative_task)

state_to_operator = state_training_cycle(
    training_pairs
)

test_grid = np.asarray(
    representative_task["test"][0]["input"]
)

test_instruction_layout = derive_state_instruction_layout(
    test_grid,
    state_to_operator,
)

prediction = test_cycle(
    test_grid,
    test_instruction_layout,
)

expected_output = np.asarray(
    representative_task["test"][0]["output"]
)

exact_match = np.array_equal(
    prediction,
    expected_output,
)

exact_match

True

In [4]:
fixed_task = challenges["00576224"]

training_pairs = get_training_pairs(fixed_task)
instruction_layout = training_cycle(training_pairs)

test_grid = np.asarray(
    fixed_task["test"][0]["input"]
)

prediction = test_cycle(
    test_grid,
    instruction_layout,
)

expected_output = np.asarray(
    fixed_task["test"][0]["output"]
)

np.array_equal(prediction, expected_output)

NameError: name 'training_cycle' is not defined